In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from google.colab import files
from sklearn.model_selection import KFold
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import Ridge

print("All libraries imported successfully!")

All libraries imported successfully!


In [2]:
uploaded = files.upload()

Saving Online Retail.xlsx to Online Retail.xlsx


In [3]:
df = pd.read_excel("Online Retail.xlsx")

print("Original dataset shape:", df.shape)
display(df.head())

Original dataset shape: (541909, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [4]:
df = df.drop_duplicates().copy()

df["InvoiceDate"] = pd.to_datetime(
    df["InvoiceDate"], errors="coerce"
)

data = df[
    (df["UnitPrice"] > 0) &
    (df["Quantity"] > 0) &
    (df["InvoiceDate"].notna()) &
    (df["StockCode"].notna())
].copy()

data["Month"] = data["InvoiceDate"].dt.month
data["Hour"] = data["InvoiceDate"].dt.hour
data["DayOfWeek"] = data["InvoiceDate"].dt.dayofweek

data["StockCode"] = data["StockCode"].astype(str)

print("Prepared dataset shape:", data.shape)
display(data.head())

Prepared dataset shape: (524878, 11)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Month,Hour,DayOfWeek
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,12,8,2
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,12,8,2
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,12,8,2
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,12,8,2
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,12,8,2


In [5]:
sample = data[
    ["UnitPrice", "Quantity", "StockCode",
     "Month", "Hour", "DayOfWeek"]
].dropna().copy()

if len(sample) > 30000:
    sample = sample.sample(
        n=30000, random_state=42
    )

T = sample["UnitPrice"].to_numpy()
Y = sample["Quantity"].to_numpy()

X = sample[
    ["StockCode", "Month", "Hour", "DayOfWeek"]
].copy()

X["StockCode"] = X["StockCode"].astype(str)

print("X shape:", X.shape)
print("T shape:", T.shape)
print("Y shape:", Y.shape)
print("Sample size:", len(sample))

X shape: (30000, 4)
T shape: (30000,)
Y shape: (30000,)
Sample size: 30000


In [6]:
preprocessor = ColumnTransformer([
    (
        "product",
        OneHotEncoder(handle_unknown="ignore"),
        ["StockCode"]
    ),
    (
        "time",
        "passthrough",
        ["Month", "Hour", "DayOfWeek"]
    )
])

kf = KFold(n_splits=2, shuffle=True, random_state=42)

T_residual = np.zeros(len(T))
Y_residual = np.zeros(len(Y))

for train_idx, test_idx in kf.split(X):
    model_t = make_pipeline(
        preprocessor,
        Ridge(alpha=10.0)
    )

    model_y = make_pipeline(
        preprocessor,
        Ridge(alpha=10.0)
    )

    model_t.fit(X.iloc[train_idx], T[train_idx])
    model_y.fit(X.iloc[train_idx], Y[train_idx])

    T_residual[test_idx] = (
        T[test_idx] - model_t.predict(X.iloc[test_idx])
    )

    Y_residual[test_idx] = (
        Y[test_idx] - model_y.predict(X.iloc[test_idx])
    )

denominator = np.mean(T_residual ** 2)

if denominator <= 1e-10:
    raise ValueError("Insufficient residual price variation.")

theta = np.mean(T_residual * Y_residual) / denominator

print("Preliminary DML coefficient:", round(theta, 6))
print("Residual price variation:", round(denominator, 6))

Preliminary DML coefficient: -0.003387
Residual price variation: 2621.593332


In [7]:
report = pd.DataFrame({
    "Metric": [
        "Sample size",
        "Unique products",
        "Treatment variable",
        "Outcome variable",
        "Control variables",
        "Preliminary DML coefficient",
        "Residual price variation"
    ],
    "Value": [
        len(sample),
        sample["StockCode"].nunique(),
        "UnitPrice",
        "Quantity",
        "StockCode, Month, Hour, DayOfWeek",
        theta,
        denominator
    ]
})

display(report)

report.to_csv(
    "day14_dml_validation_summary.csv",
    index=False
)

print("Day 14 report saved successfully!")

,Metric,Value
0,Sample size,30000
1,Unique products,3050
2,Treatment variable,UnitPrice
3,Outcome variable,Quantity
4,Control variables,"StockCode, Month, Hour, DayOfWeek"
5,Preliminary DML coefficient,-0.003387
6,Residual price variation,2621.593332


Day 14 report saved successfully!


In [8]:
files.download("day14_dml_validation_summary.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>